# MARICL-AL results
Parses `runs/Comparison.txt` and shows mean ± std across all saved runs.

In [ ]:
import re, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

RUNS_DIR  = "Runs-ACTG"
COMP_FILE = os.path.join(RUNS_DIR, "Comparison.txt")
ORACLE    = 0.0   # Perfect CATE estimation (oracle sqrt-PEHE = 0)

# ── Parse Comparison.txt ──────────────────────────────────────────────────────
with open(COMP_FILE) as f:
    raw = f.read()

# Split on separator lines of ====
blocks = [b.strip() for b in re.split(r"={10,}", raw) if "final PEHE" in b]
print(f"Found {len(blocks)} runs in {COMP_FILE}")

float_re = re.compile(r"[-+]?\d+\.\d+")
runs_data = {}   # {policy: [[pehe_r0..r5], ...]}

for block in blocks:
    for line in block.split("\n"):
        line = line.strip()
        if not line or "final PEHE" in line or line.startswith("-"):
            continue
        m = re.search(r"\s+[-+]?\d+\.\d+", line)
        if not m:
            continue
        policy = line[:m.start()].strip()
        nums   = float_re.findall(line)
        if len(nums) < 3:
            continue
        bsf = [float(x) for x in nums[:-1]]  # last num is the final PEHE (duplicate of r5)
        runs_data.setdefault(policy, []).append(bsf)

n_runs = max(len(v) for v in runs_data.values())
R      = len(next(iter(runs_data.values()))[0]) - 1
print(f"{n_runs} runs  |  {R} AL rounds  |  {len(runs_data)} policies")

# ── Compute stats ─────────────────────────────────────────────────────────────
stats = {}
for policy, trajs in runs_data.items():
    arr  = np.array(trajs)
    mean = arr.mean(axis=0)
    std  = arr.std(axis=0, ddof=1) if len(trajs) > 1 else np.zeros_like(mean)
    stats[policy] = dict(mean=mean, std=std, n=len(trajs), arr=arr)

POLICY_ORDER = [
    "MARICL-AL (XGBoost)",
    "MARICL-AL (T-Learner)",
    "MARICL-AL (CausalPFN)",
    "GP-UCB",
    "GP-EI",
    "Ens-sigma (XGB)",
    "Exploit-only",
    "Random",
    "R-Design (TSR)",
]
COLORS = {
    "MARICL-AL (XGBoost)":   "tab:blue",
    "MARICL-AL (T-Learner)": "tab:red",
    "MARICL-AL (CausalPFN)": "tab:pink",
    "GP-UCB":                "tab:green",
    "GP-EI":                 "tab:olive",
    "Ens-sigma (XGB)":       "tab:purple",
    "Exploit-only":          "tab:orange",
    "Random":                "tab:gray",
    "R-Design (TSR)":        "tab:brown",
}
STYLES = {
    "MARICL-AL (XGBoost)":   dict(lw=2.5, ls="-",  marker="o"),
    "MARICL-AL (T-Learner)": dict(lw=2.5, ls="--", marker="D"),
    "MARICL-AL (CausalPFN)": dict(lw=2.5, ls="-.", marker="P"),
}
ordered = [p for p in POLICY_ORDER if p in stats]


## Summary table — mean trajectory + final gap ± std

In [ ]:
W = max(len(p) for p in ordered)
hdr = f"{'policy':<{W}}  " + "  ".join(f"  r{i}" for i in range(R+1)) + "   final sqrt(PEHE) (mean ± std)"
print("=" * len(hdr))
print(f"SUMMARY  ({n_runs} runs)")
print("=" * len(hdr))
print(hdr)
print("-" * len(hdr))
for p in ordered:
    s    = stats[p]
    traj = "  ".join(f"{v:.3f}" for v in s["mean"])
    gap  = s["mean"][-1] - ORACLE
    print(f"{p:<{W}}  {traj}   {gap:+.3f} ± {s['std'][-1]:.3f}")
print("=" * len(hdr))


## Mean ± std trajectory + final-gap bar chart

In [ ]:
rounds = np.arange(R + 1)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
desc = "ACTG dataset"

for p in ordered:
    s   = stats[p]
    col = COLORS.get(p, "k")
    kw  = STYLES.get(p, dict(lw=1.4, ls=":", marker="."))
    axes[0].plot(rounds, s["mean"], color=col, label=p, **kw)
    if s["n"] > 1:
        axes[0].fill_between(rounds,
                              s["mean"] - s["std"],
                              s["mean"] + s["std"],
                              color=col, alpha=0.15)

axes[0].set_xlabel("AL round")
axes[0].set_ylabel("sqrt(PEHE)  — lower is better")
axes[0].set_title(f"MARICL-AL vs baselines — mean ± 1 std  ({n_runs} runs)\n"+desc)
axes[0].set_xticks(rounds)
axes[0].grid(alpha=0.3)
axes[0].legend(loc="upper right", fontsize=8)

gaps_mean = [stats[p]["mean"][-1] - ORACLE for p in ordered]
gaps_std  = [stats[p]["std"][-1]           for p in ordered]
cols      = [COLORS.get(p, "k")            for p in ordered]
sort_idx  = np.argsort(gaps_mean)[::-1]   # descending: best (smallest) at top of bar chart
axes[1].barh([ordered[i] for i in sort_idx],
              [gaps_mean[i] for i in sort_idx],
              xerr=[gaps_std[i] for i in sort_idx],
              color=[cols[i] for i in sort_idx],
              capsize=4, error_kw=dict(ecolor="k", lw=1))
axes[1].axvline(0, color="red", ls="--", alpha=0.5)
axes[1].set_xlabel("sqrt(PEHE) — lower is better  (mean ± std)")
axes[1].set_title(f"Final sqrt(PEHE) after {R} rounds  ({n_runs} runs)\n"+desc)
axes[1].grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig(os.path.join(RUNS_DIR, "summary.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {os.path.join(RUNS_DIR, 'summary.png')}")


## Individual run trajectories — MARICL models only

In [ ]:
maricl_policies = [p for p in ordered if p.startswith("MARICL-AL")]
fig, axes = plt.subplots(1, len(maricl_policies), figsize=(5 * len(maricl_policies), 4.5), sharey=True)
if len(maricl_policies) == 1:
    axes = [axes]

for ax, p in zip(axes, maricl_policies):
    col  = COLORS.get(p, "k")
    arr  = stats[p]["arr"]
    mean = stats[p]["mean"]
    for run_i, traj in enumerate(arr):
        ax.plot(rounds, traj, color=col, alpha=0.3, lw=1, marker=".", ms=4)
    ax.plot(rounds, mean, color=col, lw=2.5, marker="o", label="mean")
    ax.set_title(p.replace("MARICL-AL ", ""), fontsize=10)
    ax.set_xlabel("AL round")
    ax.set_xticks(rounds)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

axes[0].set_ylabel("sqrt(PEHE)  — lower is better")
fig.suptitle(f"Individual runs (n={n_runs}) — faint lines = single runs, bold = mean", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(RUNS_DIR, "spaghetti.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {os.path.join(RUNS_DIR, 'spaghetti.png')}")


## Win-rate — how often each model achieves the best final yield

In [ ]:
maricl_only = [p for p in ordered if p.startswith("MARICL-AL")]

# Build (n_runs x n_policies) final-PEHE matrix for MARICL models
final = np.array([[stats[p]["arr"][i][-1] for p in maricl_only]
                   for i in range(n_runs)])

wins = (final == final.min(axis=1, keepdims=True)).sum(axis=0)
print(f"{'Policy':<30}  Wins / {n_runs}  Win-rate")
print("-" * 50)
for p, w in zip(maricl_only, wins):
    print(f"{p:<30}  {w:>5}       {w/n_runs:.0%}")

# Also print per-run winner
print(f"\nPer-run winner (lowest sqrt-PEHE):")
for i in range(n_runs):
    winner = maricl_only[int(final[i].argmin())]
    print(f"  Run {i+1}: {winner}  (pehe={final[i].min():.3f})")
